In [2]:
import openai
import base64

# Set your API key
openai.api_key = "sk-..."

# Helper function to encode image as base64
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

img1_path = ".data/BUPT-CBFace-12-top-100/images/m.0_1f7bq/0.jpg"
img2_path = ".data/BUPT-CBFace-12-top-100/images/m.0qscvd0/3.jpg"

# Encode your images
image1_base64 = encode_image(img1_path)
image2_base64 = encode_image(img2_path)

# Construct the API message
response = openai.ChatCompletion.create(
    model="gpt-4o",  # Or use "gpt-4-vision-preview" if needed
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image1_base64}"}},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image2_base64}"}},
                {
                    "type": "text",
                    "text": "Do these two images show the same person? Justify your answer with identity-preserving attributes only.",
                },
            ],
        }
    ],
    max_tokens=500,
)

# Print the model's reply
print(response['choices'][0]['message']['content'])

APIRemovedInV1: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742


In [6]:
import openai
import base64
import os
openai.api_key = os.getenv("OPENAI_API_KEY")

def encode_image_to_data_url(path: str) -> str:
    with open(path, "rb") as f:
        b = f.read()
    b64 = base64.b64encode(b).decode("utf-8")
    # You may adjust mime type (png/jpg) accordingly
    return f"data:image/png;base64,{b64}"

# Encode your two images
img1 = encode_image_to_data_url(img1_path)
img2 = encode_image_to_data_url(img2_path)

# Build the request
response = openai.responses.create(
    model="gpt-5",
    input=[
        {
            "role": "user",
            "content": [
                { "type": "input_image", "image_url": img1 },
                { "type": "input_image", "image_url": img2 },
                { "type": "input_text", "text": "Do these two images show the same person? Explain using identity‑preserving attributes." },
            ]
        }
    ],
    max_output_tokens=512,
    # verbosity="medium",
    # reasoning_effort="minimal"  # you can raise this if you want deeper explanations
)

# Extract the answer
print(response.output_text)

No. They appear to be different people.

- Left: older male-presenting face, dark hair, broader nose, heavier jaw/neck, deeper facial lines.
- Right: younger female-presenting face, blonde hair, green eyes, finer nose, slimmer jawline, small beauty mark near the chin.

These stable identity features don’t match.


In [9]:
import openai
import base64
import os

openai.api_key = os.getenv("OPENAI_API_KEY")

system_prompt = "You are a helpful assistant that answers questions about images."
user_question = "Do these two images show the same person? Explain using identity‑preserving    attributes."

def encode_image_to_data_url(path: str) -> str:
    with open(path, "rb") as f:
        b = f.read()
    b64 = base64.b64encode(b).decode("utf-8")
    return f"data:image/png;base64,{b64}"

def query_gpt5_with_images(system_prompt: str, user_question: str, img1_path: str, img2_path: str):
    img1 = encode_image_to_data_url(img1_path)
    img2 = encode_image_to_data_url(img2_path)

    response = openai.responses.create(
        model="gpt-5-main",
        input=[
            {
                "role": "system",
                "content": [
                    { "type": "input_text", "text": system_prompt }
                ]
            },
            {
                "role": "user",
                "content": [
                    { "type": "input_image", "image_url": img1 },
                    { "type": "input_image", "image_url": img2 },
                    { "type": "input_text", "text": user_question }
                ]
            }
        ],
        max_output_tokens=512,
    )

    return response.output_text



system_msg = "You are a forensic biometrics expert analyzing face similarities using identity-preserving features."
user_msg = "Do these two images show the same person? Highlight similarities and dissimilarities using facial attributes."

answer = query_gpt5_with_images(system_msg, user_msg, img1_path, img2_path)
print(answer)


BadRequestError: Error code: 400 - {'error': {'message': "The requested model 'gpt-5-main' does not exist.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_found'}}

In [12]:
import openai
import base64
import os

openai.api_key = os.getenv("OPENAI_API_KEY")

system_prompt = "You are a helpful assistant that answers questions about images."
user_question = "Do these two images show the same person? Explain using identity‑preserving attributes."

def encode_image_to_data_url(path: str) -> str:
    with open(path, "rb") as f:
        b = f.read()
    b64 = base64.b64encode(b).decode("utf-8")
    return f"data:image/png;base64,{b64}"

def query_gpt5_with_images(system_prompt: str, user_question: str, img1_path: str, img2_path: str):
    img1 = encode_image_to_data_url(img1_path)
    img2 = encode_image_to_data_url(img2_path)

    response = openai.chat.completions.create(
        model="	gpt-5-chat-latest",  # Use gpt-4o or gpt-4-vision-preview if you're not explicitly using gpt-5
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": [
                    { "type": "image_url", "image_url": { "url": img1 } },
                    { "type": "image_url", "image_url": { "url": img2 } },
                    { "type": "text", "text": user_question }
                ]
            }
        ],
        max_tokens=512
    )

    return response.choices[0].message.content


answer = query_gpt5_with_images(system_prompt, user_question, img1_path, img2_path)
print(answer)

BadRequestError: Error code: 400 - {'error': {'message': 'invalid model ID', 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [13]:
import openai
import base64
import os

openai.api_key = os.getenv("OPENAI_API_KEY")

def encode_image_to_data_url(path: str) -> str:
    with open(path, "rb") as f:
        data = f.read()
    b64 = base64.b64encode(data).decode("utf‑8")
    return f"data:image/png;base64,{b64}"

def query_with_images(system_prompt: str, user_question: str, img1_path: str, img2_path: str):
    img1_url = encode_image_to_data_url(img1_path)
    img2_url = encode_image_to_data_url(img2_path)

    response = openai.responses.create(
        model="gpt-5",  # or the latest multimodal-capable model per your access
        input=[
            {
                "role": "system",
                "content": [
                    { "type": "text", "text": system_prompt }
                ]
            },
            {
                "role": "user",
                "content": [
                    { "type": "input_image", "image_url": img1_url },
                    { "type": "input_image", "image_url": img2_url },
                    { "type": "text", "text": user_question }
                ]
            }
        ],
        max_output_tokens=512,
    )

    # The official docs return a “text” output in field `output_text`
    return response.output_text


# Example usage
query_with_images(system_prompt, user_question, img1_path, img2_path)

BadRequestError: Error code: 400 - {'error': {'message': "Invalid value: 'text'. Supported values are: 'input_text', 'input_image', 'output_text', 'refusal', 'input_file', 'computer_screenshot', and 'summary_text'.", 'type': 'invalid_request_error', 'param': 'input[0].content[0].type', 'code': 'invalid_value'}}

In [33]:
import openai
import base64
import os

openai.api_key = os.getenv("OPENAI_API_KEY")

def encode_image_to_data_url(path: str) -> str:
    with open(path, "rb") as f:
        b = f.read()
    b64 = base64.b64encode(b).decode("utf-8")
    return f"data:image/png;base64,{b64}"

def query_with_images(system_prompt: str, user_question: str, img1_path: str, img2_path: str):
    img1_url = encode_image_to_data_url(img1_path)
    img2_url = encode_image_to_data_url(img2_path)

    response = openai.responses.create(
        model="gpt-5-mini",
        input=[
            {
                "role": "system",
                "content": [
                    { "type": "input_text", "text": system_prompt }
                ]
            },
            {
                "role": "user",
                "content": [
                    { "type": "input_image", "image_url": img1_url },
                    { "type": "input_image", "image_url": img2_url },
                    { "type": "input_text", "text": user_question }
                ]
            }
        ],
        max_output_tokens=5000,
        temperature=1.0,
        text={
            "format": { "type": "text" },
            "verbosity": "low",  # options: "low", "medium", "high"
        }
    )
    print(response)
    return response


system_prompt_path = "text_generators/prompts/experiment3_with_label_guided_prompt/prompt_system.txt"
user_prompt_path = "text_generators/prompts/experiment3_with_label_guided_prompt/prompt_user_nonmatch.txt"

# Load the system prompt
with open(system_prompt_path, "r") as f:
    system_prompt = f.read()

# Load the user prompt
with open(user_prompt_path, "r") as f:
    user_question = f.read() + "\n\nYour answer should not be very long and use minimal reasoning effort."


# img1_path = 

response = query_with_images(system_prompt, user_question, img1_path, img2_path)
print(response.output_text)

Response(id='resp_68d67e1104c481939cab31a73d5886f6036d751cbbec8b83', created_at=1758887441.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-mini-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_68d67e142ed4819380e1d03e75864b95036d751cbbec8b83', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_68d67e1befe8819386260ad337f6b349036d751cbbec8b83', content=[ResponseOutputText(annotations=[], text='{\n  "hypothesis": "H2_nonmatch",\n  "similarities": [\n    {"attribute": "Eye Size", "evidence": "both appear Small/Medium", "strength": 0.2},\n    {"attribute": "Cheekbone Prominence", "evidence": "both Medium prominence", "strength": 0.3}\n  ],\n  "dissimilarities": [\n    {"attribute": "Male", "evidence": "A male vs B female", "strength": 0.95},\n    {"attribute": "Asian", "evidence": "A appears Asian vs B appears White", "strength": 0.9},\n    {"attribute": "Age group", "evide

In [27]:
print(response)

Response(id='resp_68d67baf6e4081938d94050e116395de03fd589bff3a262b', created_at=1758886831.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-5-mini-2025-08-07', object='response', output=[ResponseReasoningItem(id='rs_68d67bb12a808193b91729c60b910a1303fd589bff3a262b', summary=[], type='reasoning', content=None, encrypted_content=None, status=None), ResponseOutputMessage(id='msg_68d67bd0cdb881939587a5e04798085b03fd589bff3a262b', content=[ResponseOutputText(annotations=[], text='{\n  "hypothesis": "H2_nonmatch",\n  "similarities": [\n    {\n      "attribute": "Straight",\n      "evidence": "both show straight hair texture",\n      "strength": 0.45\n    },\n    {\n      "attribute": "Partially Visible",\n      "evidence": "ears partly visible at image edges",\n      "strength": 0.35\n    }\n  ],\n  "dissimilarities": [\n    {\n      "attribute": "Male",\n      "evidence": "Image A displays male facial structure",\n      "strength": 1.00\n    },\n    {\n    